In [ ]:
import sys
from pathlib import Path

_SRC = Path.cwd().parent / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from fraud_detect import config, io, data, features, viz

warnings.filterwarnings("ignore")
viz.configure_style()

## Feature correlation analysis

## Context

with 400 + features, many derived from similar signals, redudancy is inevitable. Highly correlated features add noise, slow down training, and can destabilize linear model. this analysis identifies clusters of correlated features for pruning

## Objective 
- compute correaltion metrices for numeric features
- identify highly correlated feature paris(|r| > 0.95)
- select candidates for removal based on correalatin and importance
- quantify potential dimensionality reduction


In [1]:
# import library and load data

train = io.read_parquet(config.MERGED_TRAIN_PATH)
print(f'Data loaded: {train.shape}')


Data loaded: (590540, 434)


In [ ]:
# numeric features
numeric_cols = train.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in ['TransactionID', 'isFraud']]

print(f'Numeric features : {len(numeric_cols)}')

# sample for speed
sample = train[numeric_cols].sample(n=50000, random_state=42)

In [7]:
# V FEATURE CORRELATION
v_cols = [c for c in numeric_cols if c.startswith('V')]
print(f'v-features : {len(v_cols)}')

# compute correaltion matrix
v_corr = sample[v_cols].corr()

# find high correlation pairs
high_corr_pairs = []
for i in range(len(v_cols)):
    for j in range(i+1, len(v_cols)):
        if abs(v_corr.iloc[i, j ]) > 0.95:
            high_corr_pairs.append((v_cols[i], v_cols[j], v_corr.iloc[i,j]))


print(f'Highly correlated pairs |r| > 95 : {len(high_corr_pairs)}')
print('\nSample pairs :')
for f1, f2, r in high_corr_pairs[:10]:
    print(f'{f1} - {f2} : {r:.3f}')

v-features : 339
Highly correlated pairs |r| > 95 : 634

Sample pairs :
V10 - V11 : 0.970
V15 - V16 : 0.985
V15 - V33 : 0.956
V15 - V57 : 0.954
V15 - V94 : 0.953
V17 - V18 : 0.994
V17 - V21 : 0.959
V18 - V21 : 0.954
V21 - V22 : 0.965
V21 - V84 : 0.956


## insight : V-features correlations clusters

from the correlation anyallysis , we identify dense clusters of redudancy :
- many v-features paris exceed r > 0.95 whic indicates near perfect correaltion
- these likely com from similiar aggregation windows or realted transformations in vesta's system
- keeping all of them adds noise and training overhead without imporving AUC

pruning strategy : for each correalted pair, drop the feature with lower target correlation or higher missingness to preserve the most predictive signal

In [12]:
# identify redundant features 

# target correaltion
target_corr = sample.join(train['isFraud']).corr()['isFraud'].drop('isFraud').abs()

# for each correalted pairm mark the one with lower target correaltionfor removal
to_remove = set()
for f1, f2, r in high_corr_pairs:
    if target_corr.get(f1, 0) < target_corr.get(f2, 0):
        to_remove.add(f1)
    else:
        to_remove.add(f2)
        
print(f'Features Recomended for removal : {len(to_remove)}')
print(f'Original V-features : {len(v_cols)}')
print(f'After pruning : {len(v_cols) - len(to_remove)}')
print(f'reduction : {len(to_remove)/ len(v_cols)*100:.1f}%')

Features Recomended for removal : 130
Original V-features : 339
After pruning : 209
reduction : 38.3%


## Insight : Dimensionality Reduction Impact

from redundancy analysis:
- we identified 634 highly correlated pairs (|r| > 0.95) among V-features
- we can remove 130 features with lower target correlation from those pairs
- this represents approximately 38% reduction in V-feature space
- training time decreases proportionally with minimal AUC loss (typically less than 0.1%)

balance : aggressive pruning speeds up iteration cycles while conservative pruning preserves potential signal. start aggressive, then add features back if AUC suffers noticeably

In [ ]:
# save redundant features

io.write_csv(
    pd.Series(list(to_remove), name="feature"),
    config.REDUNDANT_FEATURE_PATH,
    index=False,
    header=True,
)
print(f'Saved redundant features list to: {config.REDUNDANT_FEATURE_PATH}')
print(f'Features to drop: {len(to_remove)}')


## key takeaways

from the feature correlation analysis:
- redundancy is substantial: we identified 634 feature pairs with |r| > 0.95 correlation
- V-features are the main source: the Vesta-engineered V1-V339 features contain many near duplicates
- simple pruning works: for each correlated pair, drop the feature with lower target correlation
- tree models tolerate redundancy: while we are pruning for efficiency, LightGBM/XGBoost handle correlated features gracefully